# Test transportation sector

In [6]:
import os
import sqlite3
import pandas as pd
import numpy as np

## Change existing capacity

In [ ]:
def change_existing_capa(db_canoe_ori, db_canoe_new,
                         existing_capa_new):

    # remove edited database if exists
    if os.path.exists(db_canoe_new):
        os.remove(db_canoe_new)
    
    # save the edited SQLite database as a new database
    connection = sqlite3.connect(db_canoe_ori)
    connection.execute(f"VACUUM INTO '{db_canoe_new}'")
    connection.close()
    
    ## update for the new canoe database
    connection = sqlite3.connect(db_canoe_new)

    ### only keep existing capacity data for Ontario
    # existing_capa_new = existing_capa_new[existing_capa_new['region'] == 'ON'].copy()

    ### remove rows with technologies in the new table from existing capacity table
    techs_new_sales = existing_capa_new['tech'].unique().tolist()

    query_remove_existing = f"""DELETE FROM ExistingCapacity
                                WHERE tech IN ({','.join(['?']*len(techs_new_sales))})"""
    
    connection.execute(query_remove_existing, techs_new_sales)
    connection.commit()

    ### insert new vehicle sales data into existing capacity table
    existing_capa_new.to_sql('ExistingCapacity', connection, if_exists='append', index=False)
    connection.commit()

    connection.close()

In [3]:
change_existing_capa(db_canoe_ori = 'Data/Test/canoe_transport_review_test1.sqlite',
                     db_canoe_new = 'Data/Test/canoe_transport_review_final_v5_1.sqlite',
                     existing_capa_new = pd.read_csv('Data/Test/Update existing capacity for all provinces/outputs/ExistingCapacity_ALL.csv'))

FileNotFoundError: [Errno 2] No such file or directory: 'Data/Test/Update existing capacity for all provinces/outputs/ExistingCapacity_ALL.csv'

## Insert efficiency and survival curves for new tech and vintage combination

In [4]:
def insert_eff_surv_new(efficiency_new, survival_curve_new,
                         db_canoe_ori, db_canoe_new):

    # filter data to only include ON
    efficiency_new = efficiency_new[efficiency_new['region'] == 'ON']
    survival_curve_new = survival_curve_new[survival_curve_new['region'] == 'ON']

    # remove edited database if exists
    if os.path.exists(db_canoe_new):
        os.remove(db_canoe_new)
    
    # save the edited SQLite database as a new database
    connection = sqlite3.connect(db_canoe_ori)
    connection.execute(f"VACUUM INTO '{db_canoe_new}'")
    connection.close()

    # build connection to the new canoe database
    connection = sqlite3.connect(db_canoe_new)

    # insert efficiency_new to table "Efficiency"
    efficiency_new.to_sql('Efficiency', connection, if_exists='append', index=False)
    connection.commit()

    # insert survival_curve_new to table "LifetimeSurvivalCurve"
    survival_curve_new.to_sql('LifetimeSurvivalCurve', connection, if_exists='append', index=False)
    connection.commit()

    connection.close()

In [5]:
insert_eff_surv_new(efficiency_new = pd.read_csv("Data/Test/Update existing capacity for all provinces/outputs/Efficiency_NEW_ALL.csv"),
                     survival_curve_new = pd.read_csv("Data/Test/Update existing capacity for all provinces/outputs/LifetimeSurvivalCurve_NEW_ALL.csv"),
                     db_canoe_ori = 'Data/Test/canoe_transport_review_final_v5_1.sqlite',
                     db_canoe_new = 'Data/Test/canoe_transport_review_final_v5_2.sqlite')

## Update LACF

In [6]:
def update_lacf(lacf_update, db_canoe_ori, db_canoe_new):

    # remove edited database if exists
    if os.path.exists(db_canoe_new):
        os.remove(db_canoe_new)
    
    # save the edited SQLite database as a new database
    connection = sqlite3.connect(db_canoe_ori)
    connection.execute(f"VACUUM INTO '{db_canoe_new}'")
    connection.close()

    # organize LACF data
    ## only keep Ontario data
    lacf_update = lacf_update[lacf_update['region'] == 'ON']

    ## only keep data during 2025 period and cross join with all periods
    lacf_update = lacf_update[lacf_update['period'] == 2025]
    lacf_update = lacf_update.drop(columns = ['period'])
    lacf_update = lacf_update.merge(pd.DataFrame({'period': [2025, 2030, 2035, 2040, 2045, 2050]}), how = 'cross')

    # build connection to the new canoe database
    connection = sqlite3.connect(db_canoe_new)

    # delete rows in LimitAnnualCapacityFactor table whose tech is in lacf_update
    techs_lacf = lacf_update['tech'].unique().tolist()
    query_remove_existing_lacf = f"""DELETE FROM LimitAnnualCapacityFactor
                                     WHERE tech IN ({','.join(['?']*len(techs_lacf))})"""
    connection.execute(query_remove_existing_lacf, techs_lacf)
    connection.commit()

    # insert updated LACF data into LimitAnnualCapacityFactor table
    lacf_update.to_sql('LimitAnnualCapacityFactor', connection, if_exists='append', index=False)
    connection.commit()

    return lacf_update

In [7]:
update_lacf(lacf_update = pd.read_csv("Data/Test/Update existing capacity for all provinces/outputs/LimitAnnualCapacityFactor_UPDATE_ALL.csv"),
            db_canoe_ori = 'Data/Test/canoe_transport_review_final_v5_2.sqlite',
            db_canoe_new = 'Data/Test/canoe_transport_review_final_v5_3.sqlite')

,region,tech,output_comm,operator,factor,period
0,ON,T_HDV_T_BEV_N,T_D_tkm_hdv_t,le,0.182983,2025
1,ON,T_HDV_T_BEV_N,T_D_tkm_hdv_t,le,0.182983,2030
2,ON,T_HDV_T_BEV_N,T_D_tkm_hdv_t,le,0.182983,2035
3,ON,T_HDV_T_BEV_N,T_D_tkm_hdv_t,le,0.182983,2040
4,ON,T_HDV_T_BEV_N,T_D_tkm_hdv_t,le,0.182983,2045
...,...,...,...,...,...,...
751,ON,T_LDV_LTP_GSL_PHEV50_N,T_D_pkm_ldv_t,ge,0.016220,2030
752,ON,T_LDV_LTP_GSL_PHEV50_N,T_D_pkm_ldv_t,ge,0.016220,2035
753,ON,T_LDV_LTP_GSL_PHEV50_N,T_D_pkm_ldv_t,ge,0.016220,2040
754,ON,T_LDV_LTP_GSL_PHEV50_N,T_D_pkm_ldv_t,ge,0.016220,2045


## Remove some combinations of technologies and vintages

In [11]:
def remove_tech_vintage(db_canoe_ori, db_canoe_new):

    # remove edited database if exists
    if os.path.exists(db_canoe_new):
        os.remove(db_canoe_new)
    
    # save the edited SQLite database as a new database
    connection = sqlite3.connect(db_canoe_ori)
    connection.execute(f"VACUUM INTO '{db_canoe_new}'")
    connection.close()

    # build connection to the new canoe database
    connection = sqlite3.connect(db_canoe_new)
    
    # remove tech "T_LDV_LTF_GSL_HEV_EX" with vintage "2015" from tables of ExistingCapacity, Efficiency, CostVariable, and SurvivalCurve
    tech_remove = 'T_LDV_LTF_GSL_HEV_EX'
    vintage_remove = 2015
    
    connection.execute(f"""DELETE FROM ExistingCapacity WHERE tech = ? AND vintage = ?""", (tech_remove, vintage_remove))
    connection.execute(f"""DELETE FROM Efficiency WHERE tech = ? AND vintage = ?""", (tech_remove, vintage_remove))
    connection.execute(f"""DELETE FROM CostVariable WHERE tech = ? AND vintage = ?""", (tech_remove, vintage_remove))
    connection.execute(f"""DELETE FROM LifetimeSurvivalCurve WHERE tech = ? AND vintage = ?""", (tech_remove, vintage_remove))
    connection.commit()

    connection.close()

In [12]:
remove_tech_vintage(db_canoe_ori = 'Data/Test/canoe_transport_review_final_v5_3.sqlite',
                    db_canoe_new = 'Data/Test/canoe_transport_review_final_v5.sqlite')

## Search sqlite tables for substring

In [10]:
def search_tables_for_substring(db_path, substring):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Get the list of tables in the database
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    
    # List to store tables containing the substring
    tables_with_substring = []
    
    # Iterate over each table
    for table in tables:
        table_name = table[0]
        
        # Get the list of columns in the table
        cursor.execute(f'PRAGMA table_info("{table_name}");')
        columns = cursor.fetchall()
        
        # Iterate over each column
        for column in columns:
            column_name = column[1]
            
            try:
                # Search for the substring in the column
                cursor.execute(f"""
                    SELECT 1 FROM {table_name}
                    WHERE {column_name} LIKE ?
                    LIMIT 1;
                """, (f"%{substring}%",))
                
                if cursor.fetchone():
                    tables_with_substring.append(table_name)
                    break  # No need to check other columns in this table
            except sqlite3.OperationalError:
                # Skip columns that can't be searched with LIKE (e.g., BLOBs)
                continue
    
    # Close the connection
    conn.close()
    
    return tables_with_substring

# Run
result = search_tables_for_substring(db_path = 'Data/Test/canoe_transport_review_test1.sqlite',
                                     substring = 'T_dsl')
result


['CapacityToActivity',
 'Commodity',
 'CostInvest',
 'CostVariable',
 'Efficiency',
 'EmissionActivity',
 'ExistingCapacity',
 'TechGroup',
 'LifetimeTech',
 'OutputDualVariable',
 'OutputNetCapacity',
 'OutputBuiltCapacity',
 'OutputRetiredCapacity',
 'OutputFlowIn',
 'OutputFlowOut',
 'OutputEmission',
 'OutputCost',
 'LimitGrowthNewCapacity',
 'LimitAnnualCapacityFactor',
 'LimitTechInputSplitAnnual',
 'LifetimeSurvivalCurve',
 'TechGroupMember',
 'Technology']